# 第 6 章 ロジスティック回帰

「0 か 1 か」という硬い予測を、シグモイド関数で「0 から 1 の確率」に置き換えます。

対応する記事: [第 6 章 ロジスティック回帰（Polyglot Notebook（F#） の言語版）](../../../docs/article/grokking-machine-learning/fsharp/ch06.md)

実装本体: `apps/grokking-ml-fsharp/src/`

## セットアップ

実装本体（`../src/GrokkingMl/`）を `#load` で読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

VS Code の [Polyglot Notebooks 拡張](https://marketplace.visualstudio.com/items?itemName=ms-dotnettools.dotnet-interactive-vscode) で開くか、Jupyter に .NET Interactive カーネルを登録して実行します。

```bash
dotnet tool install -g Microsoft.dotnet-interactive
dotnet interactive jupyter install
jupyter lab notebooks/
```

In [1]:
#load "../src/GrokkingMl/Ch06LogisticRegression.fs"

open GrokkingMl.Ch06LogisticRegression

## シグモイド関数

実数を 0 から 1 の範囲へ押し込む関数です。**大きな負の入力でも壊れない実装** になっていることを確かめます（素朴な `1/(1+exp(-x))` は Python では例外になります）。

In [2]:
for x in [ -1000.0; -5.0; -1.0; 0.0; 1.0; 5.0; 1000.0 ] do
    printfn "sigmoid(%8.1f) = %.6f" x (sigmoid x)

printfn ""
printfn "対称性 sigmoid(2) + sigmoid(-2) = %f" (sigmoid 2.0 + sigmoid -2.0)

sigmoid(

 -1000.0

) = 

0.000000

sigmoid(

...

  1000.0

) = 

1.000000

対称性 sigmoid(2) + sigmoid(-2) = 

1.000000

## 対数損失は「確信の度合い」を測る

**当たったかどうかではなく、どれくらいの確信で当たったか** を測ります。0.51 で正解しても損失は 0.67 残るので、学習はまだ進みます。

In [3]:
printfn "%10s %22s" "予測確率" "正解が 1 のときの損失"

for probability in [ 0.99; 0.9; 0.51; 0.5; 0.1; 0.01 ] do
    printfn "%10.2f %22.4f" probability (-log probability)

      予測確率

          正解が 1 のときの損失

      0.99

                0.0101

      0.90

...

                0.6931

      0.10

                2.3026

      0.01

                4.6052

## 学習

第 5 章と同じデータを使います。**初期の損失 0.6931 は `-log(0.5)`**、つまり「すべて五分五分」の状態です。第 5 章のパーセプトロン誤差が初期状態で 0 だったのと対照的です。

In [4]:
let points: Point list =
    [ [ 1.0; 0.0 ]; [ 0.0; 2.0 ]; [ 1.0; 1.0 ]; [ 1.0; 2.0 ]
      [ 1.0; 3.0 ]; [ 2.0; 2.0 ]; [ 2.0; 3.0 ]; [ 3.0; 2.0 ] ]

let labels = [ 0; 0; 0; 0; 1; 1; 1; 1 ]

let trained, losses = logisticRegression 0.1 1000 0 points labels

printfn "重み   %A" (trained.Weights |> List.map (sprintf "%.4f"))
printfn "バイアス %.4f" trained.Bias
printfn "損失   %.4f → %.4f" (List.head losses) (List.last losses)
printfn "正解率 %.2f" (accuracy trained points labels)

重み   

["2.1482"; "1.5443"]

バイアス 

-5.5948

損失   

0.6931

 → 

0.1600

正解率 

1.00

## 確率としての出力

単に分類できているだけでなく、**どちらがより確からしいかまで答えられます。**

In [5]:
List.zip points labels
|> List.iter (fun (point, label) ->
    let probability = predictProbability trained point
    let bar = String.replicate (int (probability * 40.0)) "#"
    printfn "(%.0f,%.0f) 正解=%d  %.4f %s" point[0] point[1] label probability bar)

(

1

,

0

) 正解=

...

2

) 正解=

1

0.9809

#######################################

## 試してみる: 閾値を変える

**確率が手に入ると、再学習せずに判定の厳しさを変えられます。** 第 7 章で扱う適合率と再現率のトレードオフに直結します。

In [6]:
for threshold in [ 0.2; 0.5; 0.8 ] do
    let predictions = points |> List.map (predictWith threshold trained)
    let correct = List.zip predictions labels |> List.filter (fun (p, l) -> p = l) |> List.length
    printfn "閾値 %.1f  予測 %A  正解数 %d/%d" threshold predictions correct (List.length labels)

閾値 

0.2

  予測 

[0; 0; 0; 1; 1; 1; 1; 1]

  正解数 

...

[0; 0; 0; 0; 0; 1; 1; 1]

  正解数 

7

/

8